In [ ]:
#| default_exp pythons

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory

Which interpreter a folder runs in, and the environment a child of it wants.

This module imports nothing but fastcore. Asking which interpreter a project uses costs no Jupyter
dependency.

In [ ]:
#| export
from __future__ import annotations
import os, sys
from shutil import which
from fastcore.all import L, Path, first

`kunda` runs inside a host application. `use_app` names it and the prefix its environment variables
take, so raising the kernel ceiling is `$LEELA_MAX_KERNELS`. Both are read where they are used: a
host calls `use_app` while its own module loads, which is after this one has.

In [ ]:
#| export
#: The name kunda wears, and the prefix on every environment variable it reads. `use_app` sets both.
APP = {'name': 'kunda', 'prefix': 'KUNDA_'}

def use_app(name='kunda', env_prefix='KUNDA_'):
    "Name the host application, so what kunda reads and what it says are spelled the host's way."
    APP['name'], APP['prefix'] = name, env_prefix
    return APP

def app_name():
    "What the host calls itself: `kunda`, until `use_app` says otherwise."
    return APP['name']

def env_name(name):
    "`name` under the host's prefix, so `MAX_KERNELS` is `KUNDA_MAX_KERNELS` by default."
    return APP['prefix'] + name

def app_env(name, default=None):
    "What the environment holds for `env_name(name)`, asked now rather than at import."
    return os.environ.get(env_name(name), default)

In [ ]:
#| hide
test_eq((app_name(), env_name('MAX_KERNELS')), ('kunda', 'KUNDA_MAX_KERNELS'))
os.environ['LEELA_MAX_KERNELS'] = '3'
test_eq(app_env('MAX_KERNELS', 12), 12)                    # nothing is read until the host says so
use_app('leela', 'LEELA_')
test_eq((app_name(), env_name('MAX_KERNELS'), app_env('MAX_KERNELS', 12)),
        ('leela', 'LEELA_MAX_KERNELS', '3'))
del os.environ['LEELA_MAX_KERNELS']
test_eq(use_app(), {'name': 'kunda', 'prefix': 'KUNDA_'})
test_eq((env_name('KERNEL_IDLE'), app_env('MAX_KERNELS', 12)), ('KUNDA_KERNEL_IDLE', 12))

In [ ]:
#| export
#: Directory names a project keeps its virtual environment in, best first.
VENV_DIRS = ('.venv', 'venv', 'env', '.env')
#: Where the interpreter sits inside one, on either platform.
VENV_BINS = ('bin/python', 'Scripts/python.exe')
#: The interpreters `nearest_python` looks for, as paths relative to a project folder.
VENV_PYTHONS = tuple(f'{d}/{b}' for d in VENV_DIRS for b in VENV_BINS)

#: Names a frozen host sets to point an interpreter at its own bundle.
BUNDLE_ONLY = ('PYTHONHOME', 'PYTHONPATH', 'PYTHONEXECUTABLE', '__PYVENV_LAUNCHER__', 'RESOURCEPATH')

`VENV_PYTHONS` is the cross product of the two tuples above, in preference order. It is what
`nearest_python` looks for at each level of the walk up, so `.venv` is found before `venv` and a
POSIX layout before a Windows one.

In [ ]:
VENV_PYTHONS[:4]

('.venv/bin/python',
 '.venv/Scripts/python.exe',
 'venv/bin/python',
 'venv/Scripts/python.exe')

In [ ]:
#| hide
test_eq(len(VENV_PYTHONS), len(VENV_DIRS) * len(VENV_BINS))
test_eq(VENV_PYTHONS[0], '.venv/bin/python')

`BUNDLE_ONLY` names the variables py2app and py2exe set. A frozen app inherits them from its own
launcher, and anything it spawns has to lose them.

In [ ]:
#| export
def strip_bundle(env, frozen=None):
    """`env` without a frozen host's interpreter redirection, unchanged where there is none.

    py2app and py2exe point `PYTHONHOME` and `PYTHONPATH` at the bundle so its own helper starts.
    A child that keeps them imports the bundle's standard library under another interpreter and
    fails somewhere that names nothing to do with the cause. Outside a bundle this returns what it
    was given, untouched, so a caller that claimed nothing about an environment still claims nothing.
    """
    if not (getattr(sys, 'frozen', False) if frozen is None else frozen): return env
    for name in BUNDLE_ONLY: env.pop(name, None)
    env['PYTHONUTF8'] = '1'
    return env

def clean_env():
    "This process's environment, safe to hand to a child."
    return strip_bundle(os.environ.copy(), frozen=True)

`strip_bundle` removes that redirection. Outside a bundle it returns the mapping untouched, so a
caller that claimed nothing about an environment still claims nothing.

`frozen` overrides the `sys.frozen` check, which is what makes the bundle branch testable from a
notebook that is not itself frozen.

In [ ]:
env = {'PYTHONHOME': '/App.app/Contents/Resources', 'PATH': '/usr/bin', 'HOME': '/Users/me'}
strip_bundle(dict(env), frozen=True)

{'PATH': '/usr/bin', 'HOME': '/Users/me', 'PYTHONUTF8': '1'}

`PYTHONUTF8` is set on the way out: a bundled interpreter that lost `PYTHONHOME` also loses the
encoding the launcher chose for it.

In [ ]:
test_eq(strip_bundle(dict(env), frozen=False), env)             # nothing claimed, nothing changed
out = strip_bundle(dict(env), frozen=True)
assert not (set(BUNDLE_ONLY) & set(out)), 'every redirection name is gone'
test_eq(out['PATH'], '/usr/bin')                                # and nothing else is touched
test_eq(out['PYTHONUTF8'], '1')

`clean_env` is this process's own environment put through the same strip. It passes `frozen=True`
rather than consulting `sys.frozen`, because a caller asking for a clean environment wants one
whether or not this process is bundled.

In [ ]:
#| export
def venv_env(python=None, env=None):
    "`env` (this process's, by default) with `python`'s virtual environment in front of it."
    env = strip_bundle(dict(os.environ if env is None else env))
    if not python: return env
    bindir = str(Path(python).parent)
    env['VIRTUAL_ENV'] = str(Path(bindir).parent)
    # Not `UV_PROJECT_ENVIRONMENT`. It is read wherever the process ends up rather than where it
    # started, so a `uv sync` run in another checkout syncs that project's lock into this venv and
    # prunes everything the lock does not name. uv finds the right environment from the directory.
    env.pop('UV_PROJECT_ENVIRONMENT', None)
    env['PATH'] = bindir + os.pathsep + env.get('PATH', '')
    env.pop('PYTHONHOME', None)
    return env

`venv_env` puts an interpreter's virtual environment in front of an environment. It sets
`VIRTUAL_ENV`, prepends the interpreter's directory to `PATH`, and drops `PYTHONHOME`.

`UV_PROJECT_ENVIRONMENT` is removed rather than set. uv reads it wherever the process ends up, not
where it started, so carrying it into a child would point a `uv sync` in another checkout at this
venv.

In [ ]:
e = venv_env('/repo/.venv/bin/python', env={'PATH': '/usr/bin'})
e['VIRTUAL_ENV'], e['PATH']

('/repo/.venv', '/repo/.venv/bin:/usr/bin')

In [ ]:
#| hide
test_eq(e['VIRTUAL_ENV'], '/repo/.venv')
assert e['PATH'].startswith('/repo/.venv/bin' + os.pathsep)
test_eq(venv_env(None, env={'PATH': '/usr/bin'}), {'PATH': '/usr/bin'})   # no interpreter, no claim
assert 'UV_PROJECT_ENVIRONMENT' not in venv_env('/repo/.venv/bin/python',
                                                env={'UV_PROJECT_ENVIRONMENT': '/elsewhere'})

In [ ]:
#| export
def nearest_marked(start, markers, stop=None):
    """The nearest directory at or above `start` holding one of `markers`, and the marker it holds.

    `stop` is the folder the walk must not pass: what is above it belongs to something else.
    `markers` are relative paths, so `.venv/bin/python` asks about a file and `.git` a directory.
    """
    try: start = Path(start).resolve()
    except (OSError, ValueError): return None, None
    try: stop = Path(stop).resolve() if stop else None
    except (OSError, ValueError): stop = None
    for d in (start, *start.parents):
        for m in markers:
            if (hit := d/m).exists(): return d, hit
        if stop is not None and d == stop: break
    return None, None

`nearest_marked` walks up from `start` looking for any of `markers`, and returns the directory and
the marker it found. `markers` are relative paths, so `.git` asks about a directory and
`.venv/bin/python` about a file.

`stop` is the last directory the walk may inspect. Without it the walk runs to the filesystem root
and can find something belonging to a project that has nothing to do with this one.

A `start` that cannot be resolved returns `(None, None)` rather than raising.

In [ ]:
#| hide
tmp = TemporaryDirectory(); root = Path(tmp.name)
(root/'repo'/'.git').mkdir(parents=True)
(root/'repo'/'src'/'pkg').mkdir(parents=True)

In [ ]:
d, hit = nearest_marked(root/'repo'/'src'/'pkg', ['.git'])
d.name, hit.name

('repo', '.git')

In [ ]:
#| hide
test_eq(nearest_marked(root/'repo'/'src'/'pkg', ['.git'], stop=root/'repo'/'src'), (None, None))
test_eq(nearest_marked('/does/not/exist', ['.git']), (None, None))

In [ ]:
#| export
def _venv_pythons(root):
    "Every conventional venv interpreter path under `root`, as `(venv dir, path)`. Existence unchecked."
    root = Path(root)
    return L((d, root/d/b) for d in VENV_DIRS for b in VENV_BINS)

def project_python(roots=()):
    "The first conventional project-local virtualenv interpreter among `roots`, or None."
    for root in L(roots):
        if (p := first(p for _, p in _venv_pythons(root) if p.exists())): return str(p)
    return None

def nearest_python(start, stop=None):
    "The nearest conventional venv interpreter at or above `start`, not searched past `stop`."
    p = nearest_marked(start, VENV_PYTHONS, stop)[1]
    return str(p) if p else None

Three ways of asking, over the same conventions.

`project_python` takes the first venv interpreter directly inside any of `roots`, without walking.
`nearest_python` walks up from a path and returns the first one at or above it. Both return a `str`
or `None`, never a `Path`, because what they answer is handed to `subprocess`.

In [ ]:
#| hide
def mkvenv(root, name='.venv'):
    "A venv shaped the way these functions look for one, without building a real one."
    d = Path(root)/name/'bin'; d.mkdir(parents=True, exist_ok=True)
    py = d/'python'
    py.write_text(''); py.chmod(0o755)
    return py

In [ ]:
outer = mkvenv(root/'repo')
nearest_python(root/'repo'/'src'/'pkg').replace(str(root), '/proj')   # the temp root stands in for a checkout

'/proj/repo/.venv/bin/python'

The nearest one wins. A checkout inside a monorepo has its own environment, and that is the one its
code runs in.

In [ ]:
#| hide
inner = mkvenv(root/'repo'/'src')
test_eq(nearest_python(root/'repo'/'src'/'pkg'), str(inner))
test_eq(nearest_python(root/'repo'/'src'/'pkg', stop=root/'repo'/'src'/'pkg'), None)
test_eq(project_python([root/'repo']), str(outer))
test_eq(project_python([root/'nothing'/'here']), None)
test_eq(project_python([]), None)

d2 = TemporaryDirectory(); both = Path(d2.name)
mkvenv(both, 'venv'); dot = mkvenv(both, '.venv')
test_eq(project_python([both]), str(dot))   # a project with both made `.venv` on purpose

In [ ]:
#| export
def python_for(cwd=None, stop=None, roots=(), default=None):
    """The interpreter anything spawned for `cwd` should be inside.

    The walk up from `cwd` first, stopping at `stop` — the open folder, so a venv belonging to
    something above the workspace is not borrowed. Then whatever the caller nominates as its
    default, then the first venv among `roots`. None when nothing answers, which means "this
    interpreter" to everything downstream.
    """
    if cwd and (py := nearest_python(cwd, stop)): return py
    return default or project_python(roots)

`python_for` is the question a caller actually asks: what should anything spawned for this folder
run inside. It tries the walk up from `cwd`, then the caller's `default`, then the first venv among
`roots`.

`None` is an answer, not a failure. It means this interpreter to everything downstream.

In [ ]:
#| hide
d3 = TemporaryDirectory(); proj = Path(d3.name)/'proj'; proj.mkdir()
test_eq(python_for(proj), None)                                     # nothing to find, nothing claimed
test_eq(python_for(proj, default='/usr/bin/python3'), '/usr/bin/python3')
test_eq(python_for(proj, roots=[both]), str(dot))                   # the roots, after the default
own = mkvenv(proj)
test_eq(python_for(proj), str(own))                                 # and its own beats both

In [ ]:
#| export
def _venv_roots(roots):
    "Each folder and the checkouts directly inside it: one folder of repos holds a venv each."
    for r in L(roots):
        yield Path(r)
        try: yield from sorted(d for d in Path(r).iterdir() if d.is_dir() and not d.name.startswith('.'))
        except OSError: pass

def find_pythons(roots=(), current=None, this_label='this one'):
    """Interpreters a kernel could be launched under: this one, `current`, and the venvs in reach.

    `current` is what `python_for` resolved for whatever is open, which the walk up can find deeper
    than a folder of checkouts is scanned. Listing it is what lets a picker mark it.
    """
    out, seen = L(), set()
    def add(p, label):
        p = str(p)
        if p in seen or not os.path.exists(p): return
        seen.add(p)
        out.append({'path': p, 'label': label})
    add(sys.executable, this_label)
    if current:
        d = Path(current).parents
        add(current, f'{d[2].name}/{d[1].name}' if len(d) > 2 else str(current))
    for r in _venv_roots(roots):
        for d, p in _venv_pythons(r): add(p, f'{Path(r).name}/{d}')
    for n in ('python3', 'python'):
        if (w := which(n)): add(w, f'{n} on PATH')
    return out

`find_pythons` builds the list a picker shows. It offers this interpreter first, then `current`,
then the venvs in reach of `roots`, then whatever `python3` and `python` resolve to on `PATH`.

`_venv_roots` looks in each root and in the checkouts directly inside it, because one folder of
repos holds a venv each. `current` is listed separately: `python_for` can resolve an interpreter
deeper than that scan reaches, and a picker cannot mark what is not in the list.

Nothing is offered twice, and nothing offered is missing from disk.

In [ ]:
rows = find_pythons([root/'repo'], current=str(inner))
[r['label'] for r in rows]

['this one', 'src/.venv', 'repo/.venv', 'python3 on PATH', 'python on PATH']

In [ ]:
#| hide
paths = [r['path'] for r in rows]
test_eq(rows[0], {'path': sys.executable, 'label': 'this one'})
assert str(inner) in paths, 'the resolved interpreter is offered even though the scan misses it'
test_eq(len(paths), len(set(paths)))
assert all(os.path.exists(p) for p in paths)

In [ ]:
#| hide
for t in (tmp, d2, d3): t.cleanup()